# Organic vs. Recyclable Waste — DCNN B0

Kaggle notebook to train the from-scratch reproduction model from `EXPERIMENT-PLAN.md`. It uses the audited `splits.csv`, the official Kaggle `TEST` split, and the Table 2 architecture. This is one baseline run, not the full ablation plan.

## Kaggle setup
1. Enable a GPU accelerator (one GPU is used; no DataParallel).
2. Add the Kaggle waste dataset `techsash/waste-classification-data` as an input.
3. Add a small Kaggle input dataset containing this project's `splits.csv` (from `waste-cnn/dataset/`).
4. Run all cells. Training follows the paper settings: 224×224 RGB, batch 64, Adam at 0.001, 100 epochs, no augmentation. Mixed precision is enabled on GPU to reduce memory use.

The current audit split is **provisional**: 552/2,513 official test images are flagged by exact/dHash matching, and dHash grouping includes a 1,924-image component and mixed-label groups. This notebook reports both full-test and unflagged-test metrics; the flags are not proof of semantic leakage. Outputs are written to `/kaggle/working/`.

In [ ]:
import os
os.environ['PYTHONHASHSEED'] = '42'
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'

from concurrent.futures import ThreadPoolExecutor
import hashlib
import json
import random
import time
import tempfile
from pathlib import Path, PurePosixPath

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix, classification_report)
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
torch.use_deterministic_algorithms(True, warn_only=True)

DEVICE = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
BATCH_SIZE = 64
EPOCHS = 100
LR = 1e-3
IMAGE_SIZE = 224
OUT = Path('/kaggle/working') if Path('/kaggle/working').is_dir() else Path.cwd() / 'results'
OUT.mkdir(parents=True, exist_ok=True)
print('Device:', DEVICE)
print('PyTorch:', torch.__version__)
if DEVICE.type == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))
    print('Visible GPUs:', torch.cuda.device_count(), '(this notebook uses cuda:0 only)')
    print('GPU memory (GiB):', round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1))
else:
    print('No GPU detected; this full 100-epoch model is intended for Kaggle GPU.')
CONFIG = {'seed': SEED, 'image_size': IMAGE_SIZE, 'batch_size': BATCH_SIZE,
          'epochs': EPOCHS, 'optimizer': 'Adam', 'learning_rate': LR,
          'model': 'B0_Table2', 'dropout': 0.25, 'leaky_relu_slope': 0.25,
          'weight_decay': 0.0, 'augmentation': 'none',
          'amp': DEVICE.type == 'cuda', 'splits_md5': 'filled after split validation'}
environment = {'python': __import__('sys').version, 'pytorch': torch.__version__,
               'device': str(DEVICE), 'gpu': torch.cuda.get_device_name(0) if DEVICE.type == 'cuda' else None}
(OUT / 'config.json').write_text(json.dumps(CONFIG, indent=2), encoding='utf-8')
(OUT / 'env.json').write_text(json.dumps(environment, indent=2), encoding='utf-8')


## 1. Load the audited split and locate Kaggle images

The audit CSV contains absolute paths from the Kaggle run that created it. This cell maps the suffix beginning at `DATASET/` onto the currently attached dataset root, so the CSV remains usable when Kaggle mounts it under a different input directory.

In [ ]:
EXPECTED_SPLITS_MD5 = '66c0bcf5ba56ddfe7bc0ba4d5246e7a8'
input_root = Path('/kaggle/input')
csv_candidates = sorted(input_root.rglob('splits.csv')) if input_root.exists() else []
if not csv_candidates:
    raise FileNotFoundError('Attach a Kaggle input dataset containing dataset/splits.csv.')
splits_path = csv_candidates[0]
splits_md5 = hashlib.md5(splits_path.read_bytes()).hexdigest()
assert splits_md5 == EXPECTED_SPLITS_MD5, (
    f'Unexpected splits.csv MD5 {splits_md5}; update the expected hash only after reviewing the new audit.'
)
df = pd.read_csv(splits_path)
required = {'path', 'label', 'split', 'group_id', 'test_leak'}
assert required.issubset(df.columns), f'Missing columns: {required - set(df.columns)}'
assert len(df) == 25077 and set(df['split']) == {'train', 'val', 'test'}
CONFIG['splits_md5'] = splits_md5
(OUT / 'config.json').write_text(json.dumps(CONFIG, indent=2), encoding='utf-8')
train_groups = set(df.loc[df.split == 'train', 'group_id'])
val_groups = set(df.loc[df.split == 'val', 'group_id'])
assert train_groups.isdisjoint(val_groups), 'A duplicate group crosses train/validation.'
train_groups = set(df.loc[df.split == 'train', 'group_id'])
val_groups = set(df.loc[df.split == 'val', 'group_id'])
assert train_groups.isdisjoint(val_groups), 'A duplicate group crosses train/validation.'

# Kaggle archives may contain two identical DATASET trees; use the first matching tree.
roots = sorted(p.parent for p in input_root.rglob('TRAIN')
               if p.is_dir() and (p.parent / 'TEST').is_dir()
               and (p / 'O').is_dir() and (p / 'R').is_dir())
assert roots, 'Could not find a folder containing TRAIN/O, TRAIN/R, and TEST.'
def resolve_image_path(raw_path):
    parts = PurePosixPath(str(raw_path).replace('\\', '/')).parts
    upper = [part.upper() for part in parts]
    if 'DATASET' in upper:
        tail = parts[upper.index('DATASET') + 1:]
    else:
        tail = parts[-3:]
    for root in roots:
        candidate = root.joinpath(*tail)
        if candidate.is_file():
            return str(candidate)
    raise FileNotFoundError(f'Cannot map audited path to attached dataset: {raw_path}')

df['resolved_path'] = df['path'].map(resolve_image_path)
assert df['resolved_path'].map(lambda p: Path(p).is_file()).all()
print('Split file:', splits_path)
print('splits.csv MD5:', splits_md5)
print('Image root used:', roots[0])
print(pd.crosstab(df['split'], df['label'], margins=True))
print('Test leak flags:', int(((df.split == 'test') & (df.test_leak == 1)).sum()), '/', int((df.split == 'test').sum()))

## 2. Build a one-time resized image cache

Decode and resize each image once to RGB 224×224, then reuse the memory-mapped `uint8` cache across epochs. The cache lives in temporary Kaggle storage rather than the downloadable results. Normalization statistics are computed from the training split only.

In [ ]:
train_ids = df.index[df.split == 'train'].to_numpy()
val_ids = df.index[df.split == 'val'].to_numpy()
test_ids = df.index[df.split == 'test'].to_numpy()
cache_ids = np.sort(np.concatenate((train_ids, val_ids)))
cache_path = Path(tempfile.gettempdir()) / f'waste_trainval_224_{splits_md5[:12]}.npy'
ready_path = cache_path.with_suffix('.ready')
cache_shape = (len(df), IMAGE_SIZE, IMAGE_SIZE, 3)

def resize_row(row):
    with Image.open(df.at[row, 'resolved_path']) as image:
        pixels = np.asarray(image.convert('RGB').resize(
            (IMAGE_SIZE, IMAGE_SIZE), Image.Resampling.BILINEAR))
    return row, pixels

if cache_path.exists() and ready_path.exists():
    cache = np.load(cache_path, mmap_mode='r+')
    assert cache.shape == cache_shape, f'Bad cache shape: {cache.shape}'
    print('Reusing train/validation cache:', cache_path)
else:
    cache_path.unlink(missing_ok=True)
    ready_path.unlink(missing_ok=True)
    cache = np.lib.format.open_memmap(cache_path, mode='w+', dtype=np.uint8, shape=cache_shape)
    started = time.time()
    workers = min(8, os.cpu_count() or 4)
    with ThreadPoolExecutor(max_workers=workers) as pool:
        for n, (row, pixels) in enumerate(pool.map(resize_row, cache_ids), 1):
            cache[row] = pixels
            if n % 2000 == 0 or n == len(cache_ids):
                print(f'Cached train/validation {n:,}/{len(cache_ids):,}')
    cache.flush()
    ready_path.write_text(splits_md5, encoding='ascii')
    print(f'Train/validation cache built in {(time.time() - started) / 60:.1f} min:', cache_path)

channel_sum = np.zeros(3, dtype=np.float64)
channel_sq_sum = np.zeros(3, dtype=np.float64)
pixel_count = 0
for start in range(0, len(train_ids), 128):
    batch = np.asarray(cache[train_ids[start:start + 128]], dtype=np.float32) / 255.0
    channel_sum += batch.sum(axis=(0, 1, 2))
    channel_sq_sum += np.square(batch).sum(axis=(0, 1, 2))
    pixel_count += batch.shape[0] * IMAGE_SIZE * IMAGE_SIZE
mean = (channel_sum / pixel_count).astype(np.float32)
std = np.sqrt(channel_sq_sum / pixel_count - mean ** 2).astype(np.float32)
print('Train-only channel mean:', mean.round(4))
print('Train-only channel std:', std.round(4))
print('Image dtype after normalization will be float32:', (np.ones(1, dtype=np.float32) - mean[0]).dtype)

## 3. Define the PyTorch dataset and B0 model

The layer order follows Table 2: Conv → LeakyReLU → BatchNorm; pools follow convolution 4 and 6. Initialization and BatchNorm settings mirror the Keras defaults inferred in the plan.

In [ ]:
class WasteImages(Dataset):
    def __init__(self, row_ids):
        self.row_ids = np.asarray(row_ids)
    def __len__(self):
        return len(self.row_ids)
    def __getitem__(self, item):
        row = int(self.row_ids[item])
        image = np.array(cache[row], dtype=np.float32, copy=True) / 255.0
        image = (image - mean) / std
        image = torch.from_numpy(np.ascontiguousarray(image.transpose(2, 0, 1)))
        label = torch.tensor(int(df.iloc[row].label), dtype=torch.long)
        return image, label

train_loader = DataLoader(WasteImages(train_ids), batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=DEVICE.type == 'cuda',
                          persistent_workers=True)
val_loader = DataLoader(WasteImages(val_ids), batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=2, pin_memory=DEVICE.type == 'cuda',
                        persistent_workers=True)
test_loader = DataLoader(WasteImages(test_ids), batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=2, pin_memory=DEVICE.type == 'cuda',
                         persistent_workers=True)

class DCNN(nn.Module):
    def __init__(self):
        super().__init__()
        layers = []
        channels = [3, 32, 32, 64, 64, 128, 128, 256, 256]
        for i in range(8):
            layers.extend([nn.Conv2d(channels[i], channels[i + 1], 3, padding=1, bias=True),
                           nn.LeakyReLU(negative_slope=0.25, inplace=True),
                           nn.BatchNorm2d(channels[i + 1], eps=1e-3, momentum=0.01)])
            if i in (3, 5):
                layers.append(nn.MaxPool2d(kernel_size=2, stride=2))
        self.features = nn.Sequential(*layers)
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Linear(256 * 56 * 56, 128),
            nn.LeakyReLU(negative_slope=0.25, inplace=True),
            nn.Dropout(p=0.25), nn.Linear(128, 2))
        self.apply(self._init_weights)
    @staticmethod
    def _init_weights(module):
        if isinstance(module, (nn.Conv2d, nn.Linear)):
            nn.init.xavier_uniform_(module.weight)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.BatchNorm2d):
            nn.init.ones_(module.weight)
            nn.init.zeros_(module.bias)
    def forward(self, x):
        return self.classifier(self.features(x))

model = DCNN().to(DEVICE)
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
bn_buffers = sum(b.numel() for name, b in model.named_buffers() if 'running_' in name)
print(f'Trainable parameters: {trainable_params:,}; BN running stats: {bn_buffers:,}')
assert trainable_params == 103_935_010 and bn_buffers == 1_920

## 4. Train for 100 epochs

The notebook uses one GPU (`cuda:0`) and AMP FP16 on CUDA, with FP32 fallback on CPU. It trains all 100 epochs without early stopping, records train/validation loss and metrics each epoch, and keeps the checkpoint with the highest validation accuracy (ties: lower validation loss).

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR, betas=(0.9, 0.999), eps=1e-7, weight_decay=0)
scaler = torch.cuda.amp.GradScaler(enabled=DEVICE.type == 'cuda')
history = []
best_val_acc = -1.0
best_val_loss = float('inf')
best_epoch = None
best_state = None

def epoch_metrics(labels, predictions, probabilities, loss_sum, count):
    cm = confusion_matrix(labels, predictions, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    return {
        'loss': loss_sum / count,
        'accuracy': accuracy_score(labels, predictions),
        'precision': precision_score(labels, predictions, pos_label=1, zero_division=0),
        'recall': recall_score(labels, predictions, pos_label=1, zero_division=0),
        'f1': f1_score(labels, predictions, pos_label=1, zero_division=0),
        'macro_f1': f1_score(labels, predictions, average='macro', zero_division=0),
        'auc': roc_auc_score(labels, probabilities),
        'mdr': fn / max(fn + tp, 1),
        'fdr': fp / max(fp + tp, 1),
        'fpr': fp / max(fp + tn, 1),
    }

def run_epoch(loader, training):
    model.train(training)
    all_y, all_pred, all_prob = [], [], []
    loss_sum = 0.0
    count = 0
    context = torch.enable_grad if training else torch.no_grad
    with context():
        for images, labels in loader:
            images = images.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)
            if training:
                optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=DEVICE.type == 'cuda'):
                logits = model(images)
                loss = criterion(logits, labels)
            if training:
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            probs = torch.softmax(logits.detach(), dim=1)[:, 1]
            preds = logits.detach().argmax(dim=1)
            n = labels.size(0)
            loss_sum += loss.item() * n
            count += n
            all_y.extend(labels.cpu().numpy())
            all_pred.extend(preds.cpu().numpy())
            all_prob.extend(probs.float().cpu().numpy())
    return epoch_metrics(np.asarray(all_y), np.asarray(all_pred), np.asarray(all_prob), loss_sum, count)

for epoch in range(1, EPOCHS + 1):
    started = time.time()
    train_m = run_epoch(train_loader, training=True)
    val_m = run_epoch(val_loader, training=False)
    row = {'epoch': epoch, 'lr': optimizer.param_groups[0]['lr'],
           **{f'train_{k}': v for k, v in train_m.items()},
           **{f'val_{k}': v for k, v in val_m.items()},
           'epoch_time_s': time.time() - started}
    history.append(row)
    pd.DataFrame(history).to_csv(OUT / 'history.csv', index=False)
    if (val_m['accuracy'] > best_val_acc or
        (val_m['accuracy'] == best_val_acc and val_m['loss'] < best_val_loss)):
        best_val_acc, best_val_loss = val_m['accuracy'], val_m['loss']
        best_epoch = epoch
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        torch.save({'model_state': best_state, 'epoch': epoch, 'val_metrics': val_m,
                    'splits_md5': splits_md5, 'mean': mean.tolist(), 'std': std.tolist()},
                   OUT / 'best.pt')
    print(f"Epoch {epoch:03d}/{EPOCHS} | "
          f"train loss {train_m['loss']:.4f} acc {train_m['accuracy']:.4f} "
          f"P/R {train_m['precision']:.4f}/{train_m['recall']:.4f} | "
          f"val loss {val_m['loss']:.4f} acc {val_m['accuracy']:.4f} "
          f"P/R {val_m['precision']:.4f}/{val_m['recall']:.4f} "
          f"macro-F1 {val_m['macro_f1']:.4f} | {row['epoch_time_s'] / 60:.1f} min")

assert best_state is not None
del model
if DEVICE.type == 'cuda':
    torch.cuda.empty_cache()
model = DCNN().to(DEVICE)
model.load_state_dict(best_state)
print('Best epoch:', best_epoch)
print('Best validation accuracy:', best_val_acc)

## 5. Training curves

In [ ]:
hist = pd.DataFrame(history)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, metric, title in zip(axes, ['loss', 'accuracy', 'macro_f1'], ['Loss', 'Accuracy', 'Macro-F1']):
    ax.plot(hist.epoch, hist[f'train_{metric}'], label='Train')
    ax.plot(hist.epoch, hist[f'val_{metric}'], label='Validation')
    ax.set_xlabel('Epoch'); ax.set_title(title); ax.grid(alpha=0.25); ax.legend()
plt.tight_layout()
fig.savefig(OUT / 'training_curves.png', dpi=150)
plt.show()

## 6. Final evaluation

Evaluate the restored best-validation checkpoint once on the official test set, then report a separate subset excluding rows flagged by the audit. The paper-defined FDR is `FP/(TP+FP)`; FPR is also reported separately.

In [ ]:
# Read official test images only after training and checkpoint selection are complete.
started = time.time()
workers = min(8, os.cpu_count() or 4)
with ThreadPoolExecutor(max_workers=workers) as pool:
    for n, (row, pixels) in enumerate(pool.map(resize_row, test_ids), 1):
        cache[row] = pixels
        if n % 1000 == 0 or n == len(test_ids):
            print(f'Cached test {n:,}/{len(test_ids):,}')
cache.flush()
print(f'Test cache built in {(time.time() - started) / 60:.1f} min')

@torch.no_grad()
def predict_rows(row_ids):
    loader = DataLoader(WasteImages(row_ids), batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=2, pin_memory=DEVICE.type == 'cuda', persistent_workers=True)
    model.eval()
    ys, probs = [], []
    for images, labels in loader:
        images = images.to(DEVICE, non_blocking=True)
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=DEVICE.type == 'cuda'):
            logits = model(images)
        ys.extend(labels.numpy())
        probs.extend(torch.softmax(logits.float(), dim=1)[:, 1].cpu().numpy())
    ys, probs = np.asarray(ys), np.asarray(probs)
    return ys, probs, (probs >= 0.5).astype(np.int64)

def report_metrics(name, y, prob, pred):
    cm = confusion_matrix(y, pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    result = {
        'count': int(len(y)), 'accuracy': accuracy_score(y, pred),
        'precision_recyclable': precision_score(y, pred, pos_label=1, zero_division=0),
        'recall_recyclable': recall_score(y, pred, pos_label=1, zero_division=0),
        'f1_recyclable': f1_score(y, pred, pos_label=1, zero_division=0),
        'macro_f1': f1_score(y, pred, average='macro', zero_division=0),
        'auc': roc_auc_score(y, prob), 'mdr': fn / max(fn + tp, 1),
        'fdr_paper_definition': fp / max(fp + tp, 1),
        'fpr': fp / max(fp + tn, 1),
        'confusion_matrix_tn_fp_fn_tp': cm.tolist(),
    }
    print(f'\n{name}:', json.dumps(result, indent=2))
    print(classification_report(y, pred, target_names=['Organic', 'Recyclable'], digits=4, zero_division=0))
    return result

y_test, p_test, pred_test = predict_rows(test_ids)
full_test = report_metrics('Official test', y_test, p_test, pred_test)
clean_mask = df.loc[test_ids, 'test_leak'].to_numpy() == 0
clean_test = report_metrics('Test excluding audit-flagged images',
                            y_test[clean_mask], p_test[clean_mask], pred_test[clean_mask])
pd.DataFrame({'path': df.loc[test_ids, 'resolved_path'].to_numpy(),
              'label': y_test, 'prob_recyclable': p_test, 'prediction': pred_test,
              'test_leak': df.loc[test_ids, 'test_leak'].to_numpy()}).to_csv(
                  OUT / 'predictions_test.csv', index=False)
metrics = {'best_epoch': best_epoch,
           'splits_md5': splits_md5, 'full_test': full_test, 'test_unflagged': clean_test}
(OUT / 'test_metrics.json').write_text(json.dumps(metrics, indent=2), encoding='utf-8')
torch.save({'model_state': best_state, 'config': {'image_size': IMAGE_SIZE, 'mean': mean.tolist(),
            'std': std.tolist(), 'label_map': {'organic': 0, 'recyclable': 1}},
            'best_epoch': metrics['best_epoch'], 'splits_md5': splits_md5}, OUT / 'best_weights.pt')
print('Saved model and results to', OUT)

## 7. Confusion matrix

The confusion matrix shown here is for the full official test set. The clean-subset metrics are reported above and in `test_metrics.json`.

In [ ]:
cm = confusion_matrix(y_test, pred_test, labels=[0, 1])
fig, ax = plt.subplots(figsize=(5, 4))
image = ax.imshow(cm, cmap='Blues')
ax.set(xticks=[0, 1], yticks=[0, 1], xticklabels=['Organic', 'Recyclable'],
       yticklabels=['Organic', 'Recyclable'], xlabel='Predicted', ylabel='Actual',
       title='Official test confusion matrix')
for (i, j), value in np.ndenumerate(cm):
    ax.text(j, i, str(value), ha='center', va='center')
fig.colorbar(image, ax=ax); plt.tight_layout()
fig.savefig(OUT / 'confusion_matrix_test.png', dpi=150)
plt.show()
print('Output files:', sorted(p.name for p in OUT.iterdir()))